In [1]:
import os
import joblib
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

# 1. Generate synthetic regression dataset
X, y = make_regression(n_samples=100, n_features=2, noise=0.1, random_state=42)

# 2. Train linear regression model
model = LinearRegression()
model.fit(X, y)

# 3. Create model directory before saving to avoid FileNotFoundError
os.makedirs('model', exist_ok=True)
joblib.dump(model, 'model/model.pkl')

# 4. Verify saved model by loading it back and making a test prediction
reloaded_model = joblib.load('model/model.pkl')
sample_pred = reloaded_model.predict([[0.1, 0.2]])[0]

print('Saved successfully to model/model.pkl')
print('Prediction from reloaded model:', sample_pred)

Saved successfully to model/model.pkl
Prediction from reloaded model: 23.589611712973284


In [7]:
app_code = '''from flask import Flask, request, jsonify
import joblib

# Load model ONCE at app startup (Do NOT load inside route functions)
model = joblib.load('model/model.pkl')
app = Flask(__name__)


@app.route('/predict', methods=['POST'])
def predict_post():
    data = request.get_json(silent=True) or {}
    features = data.get('features')

    # Validate that features exist and contain exactly 2 numbers
    if not isinstance(features, list) or len(features) != 2 or not all(isinstance(x, (int, float)) for x in features):
        return jsonify({'error': 'Invalid input. "features" must be a list of 2 numbers.'}), 400

    try:
        # Predict expects a 2D list: [[f1, f2]]
        prediction = float(model.predict([features])[0])
        return jsonify({'prediction': prediction})
    except Exception as e:
        return jsonify({'error': str(e)}), 400


@app.route('/predict/<f1>/<f2>', methods=['GET'])
def predict_get(f1, f2):
    # Convert path parameters from string to float
    try:
        f1_val = float(f1)
        f2_val = float(f2)
    except (ValueError, TypeError):
        return jsonify({'error': 'Path parameters f1 and f2 must be valid numbers.'}), 400

    try:
        prediction = float(model.predict([[f1_val, f2_val]])[0])
        return jsonify({'prediction': prediction})
    except Exception as e:
        return jsonify({'error': str(e)}), 400


if __name__ == '__main__':
    app.run(port=5000)
'''

# Write app code to app.py file on disk
with open('app.py', 'w') as f:
    f.write(app_code)

print('app.py written successfully.')

app.py written successfully.


In [8]:
import os

# Launch Flask server in an external terminal window
os.system("start cmd /k python app.py")
print("Flask server launched in an external terminal. Wait a few seconds before sending requests.")

Flask server launched in an external terminal. Wait a few seconds before sending requests.


In [9]:
import requests

BASE_URL = 'http://127.0.0.1:5000'

try:
    # 1. Test POST endpoint with valid JSON payload
    r_post = requests.post(BASE_URL + '/predict', json={'features': [0.1, 0.2]}, timeout=5)
    print('POST /predict status:         ', r_post.status_code, '| Response:', r_post.text.strip())

    # 2. Test GET endpoint with valid URL parameters
    r_get = requests.get(BASE_URL + '/predict/0.1/0.2', timeout=5)
    print('GET  /predict/0.1/0.2 status: ', r_get.status_code, '| Response:', r_get.text.strip())

    # 3. Test GET endpoint with bad parameter (deliberately invalid float 'abc')
    r_bad = requests.get(BASE_URL + '/predict/abc/0.2', timeout=5)
    print('GET  /predict/abc/0.2 status: ', r_bad.status_code, '| Response:', r_bad.text.strip())

except requests.exceptions.ConnectionError:
    print('Error: Could not connect to Flask server at http://127.0.0.1:5000.')
    print('Please ensure Cell 3 was executed and the external terminal window is running.')

POST /predict status:          200 | Response: {"prediction":23.589611712973284}
GET  /predict/0.1/0.2 status:  200 | Response: {"prediction":23.589611712973284}
GET  /predict/abc/0.2 status:  400 | Response: {"error":"Path parameters f1 and f2 must be valid numbers."}
